# 0. Setup & Orientation

*Establish scope, set up the environment, and align on expectations before any code runs.*

---

This lab picks up where the [EscalationLab](https://github.com/FrankLaVigne/EscalationLab) left off. That lab built a RAG pipeline, measured its failures, and applied every improvement available within a passive architecture: better chunking, Best-of-N sampling, fine-tuning. Some failures survived all of it.

This lab explains why those failures survived, and introduces the control structure that resolves them: the **agent loop**.

Before opening any notebooks, read this section to understand what we are building, why, and how the environment needs to be configured.

## 0.1 What This Session Is (and Is Not)

This session **is**:

- A guided walkthrough of agentic control structures around RAG pipelines
- Focused on the architectural gap that passive RAG cannot close
- A direct continuation of the Escalation Lab's evaluation results
- Designed to teach *when and why* to add agentic behavior, not just how

This session is **not**:

- A framework tutorial (no LangChain, no LlamaIndex)
- A fine-tuning or prompt engineering workshop
- A general-purpose agent-building exercise
- A "try every feature" demo

**Set expectations:**

- One scenario, one execution path
- Same model, same retriever, same data as the Escalation Lab
- The only thing that changes is the control structure

> **Say explicitly:** "Today is about architecture, not heroics."

## 0.2 The Mental Model for Today

The governing principle carries forward from the Escalation Lab:

> **Escalation of effort must be justified by evidence.**

The Escalation Lab followed this progression:

```
Baseline → Data Quality → Retrieval → Evaluation → Decision
```

This lab picks up at the decision point. The evidence says: 2 of 10 questions still fail after every passive-architecture improvement. The failures are not random. They fall into three categories:

- **Irrelevant retrieval**: the retriever returned chunks that don't contain the answer
- **Implicit reasoning**: the answer requires combining facts the model didn't connect
- **Out-of-scope**: the answer doesn't exist in the corpus

A passive pipeline cannot distinguish these cases. It retrieves, answers, and moves on. The agent loop adds the ability to **inspect**, **decide**, and **recover**.

> *"If you already know the answer is 'add an agent,' you're skipping the part where most projects fail: understanding why the current architecture can't solve the problem."*

## 0.3 What Success Looks Like

By the end of this lab, participants should be able to:

- Classify RAG failures as retrieval problems, reasoning problems, or scope problems
- Explain why a passive pipeline cannot recover from these failures
- Build a retrieval evaluator that judges context quality before answering
- Implement query rewriting as a recovery strategy
- Construct an agent loop that retrieves, evaluates, decides, and recovers
- Articulate when agentic RAG is justified and when it is not

Success is **not**:

- Running every cell successfully
- Achieving perfect scores on all 10 questions
- Memorizing the agent loop code

## 0.4 How the Lab Will Run

Set these expectations **before** anyone opens a notebook.

- This lab is delivered as a **guided Jupyter notebook**
- Code and explanation are **interleaved**
- Participants are **not expected to write code from scratch**
- Pre-generated outputs exist for **every major step**

**Facilitator guidance:**

- If a cell takes more than ~2-3 minutes to run, **move on** (tolerance will vary by cell)
- Use pre-built outputs without apology
- Focus discussion on *interpretation*, not execution

---

## 0.5 Ground Rules

**One path forward**: We follow a single canonical workflow today.

**Questions are welcome; tangents are parked**: Interesting side topics are noted but deferred.

**No premature complexity**: We don't add agentic behavior until the evidence shows passive RAG has failed.

**Evidence beats instinct**: "It feels like it needs an agent" is not a sufficient reason to build one.

## 0.6 Setting the Tone

> "This lab reflects how successful AI engagements actually unfold: you don't reach for agents because they're exciting. You reach for them because you've measured a failure that simpler architectures cannot fix."

## 0.7 Setting Up the Workbench

Before any code runs, the environment has to exist. This section walks through standing up your Red Hat AI workbench from scratch. Every decision made here (storage size, image selection, repo structure) directly affects what is possible in the lab.

### 0.7.1 Log into the Lab

Using your Red Hat SSO credentials, log into the OpenShift AI dashboard at the URL provided by your instructor. Once authenticated, you will land on the Red Hat OpenShift AI home screen.

### 0.7.2 Create a Project

From the dashboard, navigate to **Data Science Projects** and click **Create Project**. Name the project `continuum-cluster`. This project is the namespace that will contain your workbench, storage, and all associated resources for the lab series.

### 0.7.3 Create a Workbench

Inside your new project, click **Create Workbench**. Configure it as follows:

* **Name:** `continuum-workbench`
* **Image:** Jupyter | PyTorch | CUDA | Python 3.12
* **Container size:** Medium (6 CPU / 24 GiB)
* **Accelerator:** NVIDIA GPU L40S (1)

### 0.7.4 Increase Storage Size

The default storage allocation is not sufficient for this lab. The Escalation Lab repo contains large model files tracked with Git LFS, and intermediate outputs consume significant space. Before finalizing the workbench, increase the persistent storage to at least `50 GiB`.

### 0.7.5 Launch Workbench When Ready

Click **Create Workbench**. The status will show as *Starting*. Wait until the status changes to *Running*, then click **Open** to launch JupyterLab in a new tab. This typically takes 1–3 minutes on first launch.

## 0.8 Clone the Repos

Inside JupyterLab, open a terminal from the Launcher tab (**File > New > Terminal**). All remaining setup steps in this section are run here, not in a notebook cell.

### 0.8.1 Install Git LFS

The Escalation Lab repository uses Git LFS to store large files, including the source PDFs used for ingestion. Without LFS initialized, cloning will pull pointer files instead of the actual content.

```bash
git lfs install
```

> **Note:** On some lab environments, `git lfs install` may fail with a "command not found" error. If it fails, skip this step and continue; a fallback is provided to download files directly.

### 0.8.2 Clone the Escalation Lab (if not already done)

If you completed the Escalation Lab, you already have this repo. If not, clone it now; the pre-built evaluation results in this lab were generated from its pipeline.

```bash
git clone https://github.com/FrankLaVigne/EscalationLab.git
```

### 0.8.3 Clone the Agentic RAG Lab

```bash
git clone https://github.com/FrankLaVigne/AgenticRag.git
```

### 0.8.4 Verify the Clone

```bash
ls AgenticRag/
```

You should see:

```
00_Setup/  01_WhyPassiveRAGBreaks/  02_TheAgentLoop/  prebuilt/  README.md
```

## 0.9 Environment Variables

The notebooks connect to a Model-as-a-Service (MaaS) endpoint running IBM Granite models. You need three environment variables set before running any notebook cells.

Create a `.env` file in the repo root (this file is gitignored):

```bash
cat > AgenticRag/.env << 'EOF'
MAAS_API_KEY=your-api-key
MAAS_BASE_URL=https://your-maas-endpoint/v1
MAAS_MODEL_ID=granite-3-2-8b-instruct
EOF
```

Your instructor will provide the API key and endpoint URL.

| Variable | Description |
|----------|-------------|
| `MAAS_API_KEY` | API key for the MaaS endpoint |
| `MAAS_BASE_URL` | Base URL for the MaaS endpoint (include `/v1`) |
| `MAAS_MODEL_ID` | Model identifier (default: `granite-3-2-8b-instruct`) |

## 0.10 Verify the Environment

Run the cell below to confirm that the environment is configured correctly. If any check fails, revisit the steps above before continuing to Section 1.

In [ ]:
import os
import json

# Load .env if present
env_path = os.path.join("..", ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip()
    print("Loaded .env file")
else:
    print("No .env file found. Using existing environment variables")

# Check required variables
checks = {
    "MAAS_API_KEY": os.environ.get("MAAS_API_KEY"),
    "MAAS_BASE_URL": os.environ.get("MAAS_BASE_URL"),
    "MAAS_MODEL_ID": os.environ.get("MAAS_MODEL_ID", "granite-3-2-8b-instruct"),
}

print("\nEnvironment Check")
print("=" * 40)
all_ok = True
for var, val in checks.items():
    status = "OK" if val else "MISSING"
    display = val[:20] + "..." if val and len(val) > 20 else val
    print(f"  {var:<20} {status:<10} {display or ''}")
    if not val:
        all_ok = False

# Check prebuilt data
eval_path = os.path.join("..", "prebuilt", "eval_results.json")
if os.path.exists(eval_path):
    with open(eval_path) as f:
        data = json.load(f)
    print(f"\n  eval_results.json    OK         {len(data['results'])} questions loaded")
else:
    print(f"\n  eval_results.json    MISSING    Expected at {eval_path}")
    all_ok = False

print("\n" + ("All checks passed. Ready for Section 1." if all_ok else "Fix the issues above before continuing."))

---

## What Comes Next

Section 1 re-examines the Escalation Lab's evaluation results and classifies the remaining failures. Understanding *what kind* of failure each one is determines whether the agent loop can fix it.

Move to `01_WhyPassiveRAGBreaks/01_Why_Passive_RAG_Breaks.ipynb`.